Classification Sweep Job-Array Worker

Each SLURM array task runs all graph-filter percentages for one topic and account-relation configuration. Results are merged safely into common CSV files.

# Imports

In [ ]:
key_authors_graph_definition_only = True
%run -i 3_key_authors_graph.ipynb

In [ ]:
import fcntl
import os
import sys
from pathlib import Path
from multiprocessing import Lock
import gc

from itertools import combinations
import re

import pandas as pd
import pyarrow.parquet as pq

# Parameters

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

In [ ]:
input_data_path = project_root / "results/Labeled_Datasets/Ecuador/Ecuador_part_all/step_2_BERTopic_clustering/Ecuador_part_all_step_2.gzip.parquet"
classification_task_id = None
top_percentages = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
models = ["random_forest", "xgboost", "logistic_regression"]

In [ ]:
topic_cols_pattern = re.compile(
    r"^(?:K_Means|BERTopic)_topic(?!.*_top_words$)"
)
account_id_col = "accountid"
is_io_control_col = "is_control"
author_key_score_col = "topic_author_key_score"
entity_cols_lst = ["hashtags", "account_mentions", "ner_entities"]
key_authors_mode = "per_topic"
is_directed_graphs = False
random_state = 42
cv_jobs = 5

# Utils

## File system

In [ ]:
input_data_path = Path(input_data_path).expanduser()
output_folder_path = input_data_path.parent.parent / "step_3_key_authors_graph/classification_sweep"
output_folder_path.mkdir(parents=True, exist_ok=True)

In [ ]:
RESULT_COLUMNS = [
    "topic_col",
    "top_percentage",
    "account_relations_configuration",
    "indicators_configuration",
    "model",
    "indicators_runtime_seconds",
    "n_graph_authors",
    "n_all_authors",
    "graph_authors_auc",
    "graph_authors_f1",
    "graph_authors_auc_pr",
    "graph_authors_recall",
    "graph_authors_precision",
    "all_authors_auc",
    "all_authors_f1",
    "all_authors_auc_pr",
    "all_authors_recall",
    "all_authors_precision",
]
RESULT_KEY_COLUMNS = [
    "topic_col",
    "top_percentage",
    "account_relations_configuration",
    "indicators_configuration",
    "model",
]
ISSUE_COLUMNS = [
    "topic_col",
    "top_percentage",
    "account_relations_configuration",
    "error_type",
    "error",
]
ISSUE_KEY_COLUMNS = [
    "topic_col",
    "top_percentage",
    "account_relations_configuration",
]

In [ ]:
def merge_rows_into_shared_csv(
    rows_df: pd.DataFrame,
    output_path: str | Path,
    columns: list[str],
    key_columns: list[str],
) -> pd.DataFrame:
    """Merge rows into a shared CSV under an exclusive file lock.

    Args:
        rows_df: New rows produced by the current array task.
        output_path: Common CSV file updated by every task.
        columns: Required output-column order.
        key_columns: Columns identifying rows across task retries.

    Returns:
        The complete, sorted, and deduplicated shared DataFrame.

    Raises:
        ValueError: If new or existing rows do not match the expected schema.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    lock_path = output_path.with_suffix(output_path.suffix + ".lock")
    temporary_path = output_path.with_name(
        f".{output_path.name}.{os.getpid()}.tmp"
    )

    missing_new_columns = sorted(set(columns) - set(rows_df.columns))
    if missing_new_columns:
        raise ValueError(f"New result rows are missing columns: {missing_new_columns}")
    new_rows_df = rows_df.loc[:, columns].copy()

    with open(lock_path, "w", encoding="utf-8") as lock_file:
        fcntl.flock(lock_file, fcntl.LOCK_EX)
        try:
            if output_path.exists():
                existing_df = pd.read_csv(output_path)
                missing_existing_columns = sorted(
                    set(columns) - set(existing_df.columns)
                )
                if missing_existing_columns:
                    raise ValueError(
                        "Existing shared CSV is missing columns: "
                        f"{missing_existing_columns}"
                    )
                existing_df = existing_df.loc[:, columns]
            else:
                existing_df = pd.DataFrame(columns=columns)

            if existing_df.empty:
                merged_df = new_rows_df
            else:
                merged_df = pd.concat(
                    [existing_df, new_rows_df], ignore_index=True
                )
            merged_df = (
                merged_df.drop_duplicates(subset=key_columns, keep="last")
                .sort_values(key_columns)
                .reset_index(drop=True)
            )
            merged_df.to_csv(temporary_path, index=False)
            os.replace(temporary_path, output_path)
        finally:
            temporary_path.unlink(missing_ok=True)
            fcntl.flock(lock_file, fcntl.LOCK_UN)

    return merged_df

## Task ID

In [ ]:
task_id_value = (
    classification_task_id
    if classification_task_id is not None
    else os.getenv("SLURM_ARRAY_TASK_ID")
)
if task_id_value is None:
    raise RuntimeError(
        "Set SLURM_ARRAY_TASK_ID or inject classification_task_id with Papermill."
    )

try:
    task_id = int(task_id_value)
except (TypeError, ValueError) as exc:
    raise ValueError(f"Classification task ID must be an integer: {task_id_value!r}") from exc

## Logs

In [ ]:
report_folder_path = output_folder_path / "progress_reports"
report_folder_path.mkdir(parents=True, exist_ok=True)
REPORT_PATH = report_folder_path / f"progress_report_task_id_{task_id}.txt"
if REPORT_PATH.exists():
    REPORT_PATH.unlink()

JOB_REPORT_PATH = report_folder_path / "jobs_reports" / f"progress_report_{JOB_ID}_task_id_{task_id}.txt"
JOB_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_LOCK = Lock()


def write_max_memory_report() -> None:
    """Write the maximum memory used by the task."""
    _, peak_bytes = get_memory_usage_bytes()
    write_report(f"MAXIMUM MEMORY USED: {format_memory(peak_bytes)}")

In [ ]:
write_hardware_report(report_folder_path)

In [ ]:
write_report(f"####################### STRAT TASK {task_id} #######################")
write_report(f"####################### STRAT TASK {task_id} #######################")

write_report(f"Input path: {input_data_path}")
write_report(f"Output folder: {output_folder_path}")

# Configurations

## Parameters

In [ ]:
all_indicators = [
    "degree_centrality",
    "harmonic_centrality",
    "eigenvector_centrality",
    "betweenness_centrality",
    "pagerank",
    "core_number",
    "clustering",
    "louvain_community",
    "key_score_sum",
    "key_score_dominance",
    "posting_dominance",
    "top_topic",
]
indicators_configuration = {
    "All (Unweighted Clustering, No Square Clustering)": all_indicators,
    "Core Number Only": ["core_number"],
    "All but Betweenness and Harmonic (Unweighted Clustering, No Square Clustering)": [
        indicator
        for indicator in all_indicators
        if indicator not in ["betweenness_centrality", "harmonic_centrality"]
    ],
}

In [ ]:
# (source column, destination column)
similarity_account_relations = [
    ("hashtags_reuse_12H_accountids", "accountid"),
    ("hashtags_reuse_3D_accountids", "accountid"),
    ("account_mentions_reuse_12H_accountids", "accountid"),
    ("account_mentions_reuse_3D_accountids", "accountid"),
]
text_similarity_account_relations = [
    ("ner_entities_reuse_12H_accountids", "accountid"),
    ("ner_entities_reuse_3D_accountids", "accountid"),
    ("posting_same_topic_BERTopic_topic_256_12H_accountids", "accountid"),
    ("posting_same_topic_BERTopic_topic_256_3D_accountids", "accountid"),
]
interactions_account_relations = [
    ("in_reply_to_accountid", "accountid"),
    ("account_mentions", "accountid"),
    ("reposted_accountid", "accountid"),
]



account_relations = {
    "similarity": similarity_account_relations,
    "text_similarity": text_similarity_account_relations,
    "interactions": interactions_account_relations,
}

## All Possible Configuration

### Relation Combinations

In [ ]:
def build_relation_combinations(
    relation_groups: dict[str, list[tuple[str, str]]],
) -> dict[str, list[tuple[str, str]]]:
    """Build every non-empty combination of account-relation groups.

    Args:
        relation_groups: Named groups of source and destination columns.

    Returns:
        Combined relations keyed by their plus-separated group names.
    """
    relation_combinations = {}
    group_names = list(relation_groups)
    for combination_size in range(1, len(group_names) + 1):
        for selected_group_names in combinations(group_names, combination_size):
            configuration_name = "+".join(selected_group_names)
            combined_relations = [
                relation
                for group_name in selected_group_names
                for relation in relation_groups[group_name]
            ]
            relation_combinations[configuration_name] = list(
                dict.fromkeys(combined_relations)
            )
    return relation_combinations



relation_combinations = build_relation_combinations(account_relations)

### Topic Columns

In [ ]:
input_schema = pq.read_schema(input_data_path)
topic_cols = [
    column for column in input_schema.names if topic_cols_pattern.match(column)
]
if not topic_cols:
    raise ValueError(f"No classification topic columns found in {input_data_path}")

### All Configuration

In [ ]:
all_configuration_combinations = [
    (current_topic_col, configuration_name, relations)
    for current_topic_col in topic_cols
    for configuration_name, relations in relation_combinations.items()
]

percentage_experiment_count = (
    len(all_configuration_combinations) * len(top_percentages)
)
print(
    f"Found {len(all_configuration_combinations)} topic-relation tasks "
    f"covering {percentage_experiment_count} percentage experiments."
)

## Current Configuration

### Selected Configuration

In [ ]:
if not 1 <= task_id <= len(all_configuration_combinations):
    raise IndexError(
        f"Classification task ID {task_id:,} is outside the valid 1-"
        f"{len(all_configuration_combinations):,} range."
    )

In [ ]:
configuration_combination = all_configuration_combinations[task_id - 1]
selected_topic_col, selected_relation_name, selected_relations = (
    configuration_combination
)
write_report(
    f"Task {task_id:,}/{len(all_configuration_combinations):,}: "
    f"topic={selected_topic_col}, relations={selected_relation_name}, "
    f"top_percentages={top_percentages}"
)

### Checkpoint

In [ ]:
results_path = output_folder_path / "classification_sweep_results.csv"
expected_result_keys = {
    (float(percentage), indicator_name, model_name)
    for percentage in top_percentages
    for indicator_name in indicators_configuration
    for model_name in models
}
completed_result_keys = set()
if results_path.exists():
    results_df = pd.read_csv(results_path, usecols=RESULT_KEY_COLUMNS)
    current_rows = results_df[
        (results_df["topic_col"] == selected_topic_col)
        & (results_df["account_relations_configuration"] == selected_relation_name)
    ]
    completed_result_keys = set(
        zip(
            current_rows["top_percentage"].astype(float),
            current_rows["indicators_configuration"],
            current_rows["model"],
        )
    )

configuration_is_complete = expected_result_keys <= completed_result_keys
print(
    f"Configuration already completed: {configuration_is_complete} "
    f"({len(completed_result_keys):,}/{len(expected_result_keys):,} rows)"
)

# Load Data

In [ ]:
def load_json_list_with_shared_strings(value: object) -> list:
    """Decode a JSON list and reuse identical string objects.

    Args:
        value: JSON array string or an already-decoded collection.

    Returns:
        Decoded list with repeated strings interned to reduce memory use.
    """
    return [
        sys.intern(item) if isinstance(item, str) else item
        for item in load_json_list(value)
    ]


if not configuration_is_complete:
    required_sweep_columns = list(
        dict.fromkeys(
            [
                account_id_col,
                is_io_control_col,
                selected_topic_col,
                *entity_cols_lst,
                *(
                    column
                    for relation in selected_relations
                    for column in relation
                ),
            ]
        )
    )
    missing_sweep_columns = sorted(
        set(required_sweep_columns) - set(input_schema.names)
    )
    if missing_sweep_columns:
        raise KeyError(f"Missing classification sweep columns: {missing_sweep_columns}")


    print(f"required_sweep_columns: {required_sweep_columns}")

In [ ]:
if not configuration_is_complete:
    write_report(f"Loading df with columns: {required_sweep_columns}")
    df = pd.read_parquet(input_data_path, columns=required_sweep_columns).reset_index(
        drop=True
    )
    write_report(f"Loaded df with {len(df):,} rows and {len(df.columns):,} columns")

In [ ]:
if not configuration_is_complete:
    json_list_columns = [
        column
        for column in required_sweep_columns
        if column in entity_cols_lst
        or "_reuse_" in column
        or column.startswith("posting_same_topic_")
    ]
    for column in json_list_columns:
        loader = (
            load_json_list_with_shared_strings
            if column.endswith("_accountids")
            else load_json_list
        )
        df[column] = df[column].map(loader)
    df[account_id_col] = df[account_id_col].astype(str).map(sys.intern)

    io_authors_set = get_io_authors_by_ratio(
        df,
        author_col=account_id_col,
        is_io_control_col=is_io_control_col,
    )

# Classification Analysis

## Preparation

In [ ]:
if not configuration_is_complete:
    post_score_col = f"post_tfidf_{selected_topic_col}"
    current_author_score_col = f"{author_key_score_col}_{selected_topic_col}"
    compute_post_tfidf(
        df,
        selected_topic_col,
        entity_cols_lst,
        post_score_col=post_score_col,
    )
    author_topic_key_score_df = compute_author_key_score(
        df=df,
        topic_col=selected_topic_col,
        author_col=account_id_col,
        post_score_col=post_score_col,
        author_score_col=current_author_score_col,
    )

In [ ]:
if not configuration_is_complete:
    selected_relation_columns = {
        column for relation in selected_relations for column in relation
    }
    unused_entity_columns = [
        column
        for column in entity_cols_lst
        if column not in selected_relation_columns
    ]
    df.drop(columns=unused_entity_columns, inplace=True)
    gc.collect()
    print(f"Dropped unused entity columns: {unused_entity_columns}")

In [ ]:
if not configuration_is_complete:
    _init_classification_sweep_worker(
        df=df,
        author_topic_scores={selected_topic_col: author_topic_key_score_df},
        io_authors_set=io_authors_set,
        indicators_configuration=indicators_configuration,
        models=models,
        author_col=account_id_col,
        author_score_cols={selected_topic_col: current_author_score_col},
        key_authors_mode=key_authors_mode,
        is_directed_graphs=is_directed_graphs,
        random_state=random_state,
        cv_jobs=cv_jobs,
        output_path=output_folder_path,
    )

## Run

In [ ]:
skipped_path = output_folder_path / "classification_sweep_skipped.csv"
failures_path = output_folder_path / "classification_sweep_failures.csv"


def save_task_issues(task_issues: list[dict]) -> tuple[int, int]:
    """Merge percentage-level issues into the shared issue files.

    Args:
        task_issues: Worker issues containing an ``is_skippable`` flag.

    Returns:
        Number of skipped and failed percentage experiments.
    """
    skipped_issues = [
        issue for issue in task_issues if issue["is_skippable"]
    ]
    failed_issues = [
        issue for issue in task_issues if not issue["is_skippable"]
    ]
    for issues, issue_path in (
        (skipped_issues, skipped_path),
        (failed_issues, failures_path),
    ):
        if issues:
            merge_rows_into_shared_csv(
                pd.DataFrame(issues, columns=ISSUE_COLUMNS),
                issue_path,
                ISSUE_COLUMNS,
                ISSUE_KEY_COLUMNS,
            )
    return len(skipped_issues), len(failed_issues)

In [ ]:
rows = []
issues = []
task_exception = None
if configuration_is_complete:
    write_report(f"Task {task_id:,} is already complete; skipping.")
else:
    try:
        rows, issues = _run_classification_for_top_percentages(
            top_percentages,
            selected_relation_name,
            selected_topic_col,
            selected_relations,
        )
    except Exception as exc:
        task_exception = exc
        is_skippable = (
            isinstance(exc, ValueError)
            and str(exc).startswith(
                CLASSIFICATION_SKIPPABLE_ERROR_PREFIXES
            )
        )
        issues = [{
            "topic_col": selected_topic_col,
            "top_percentage": percentage,
            "account_relations_configuration": selected_relation_name,
            "error_type": type(exc).__name__,
            "error": repr(exc),
            "is_skippable": is_skippable,
        } for percentage in top_percentages]

task_results_df = pd.DataFrame(rows, columns=RESULT_COLUMNS)
classification_sweep_df = merge_rows_into_shared_csv(
    task_results_df,
    results_path,
    RESULT_COLUMNS,
    RESULT_KEY_COLUMNS,
)
skipped_count, failure_count = save_task_issues(issues)
display(task_results_df.head())
write_report(
    f"Task {task_id:,} added {len(task_results_df):,} rows to {results_path}. "
    f"The shared file now contains {len(classification_sweep_df):,} rows. "
    f"Skipped percentages: {skipped_count:,}; "
    f"failed percentages: {failure_count:,}."
)

if failure_count:
    write_max_memory_report()
    if task_exception is not None:
        raise task_exception
    raise RuntimeError(
        f"Classification task {task_id:,} failed for "
        f"{failure_count:,} of {len(top_percentages):,} percentages."
    )

# End

In [ ]:
write_report("#################### COMPLETE  ####################")
write_report("#################### COMPLETE  ####################")

write_max_memory_report()